# Core Dataset Multi-Liga - EDA

Análisis Exploratorio de Datos (EDA) estructural comparativo de las tres competiciones (`2014-15` a `2023-24`) a partir de los datasets raw de partidos procedentes de `football-data.co.uk`.

## Objetivos

- Evaluar la homogeneidad estructural entre competiciones (dimensiones, esquema y tipos de datos).
- Definir el **core dataset unificado** común a las tres ligas.
- Analizar la completitud de datos y distribución de valores nulos por competición.
- Validar integridad estructural (consistencia FTR/HTR, duplicados, valores negativos).
- Determinar la viabilidad de un pipeline de limpieza unificado y estrategia de modelado multi-liga.

## Estructura del Notebook

El análisis se organiza en las siguientes secciones:

**0. Entorno y configuración**  
Carga de librerías y definición de rutas del proyecto.

**1. Lectura y organización de datos**  
Carga de los datasets core procesados de cada liga.

**2. Comparación de esquemas**  
Comparación de dimensiones, definición del core unificado y verificación de consistencia en tipos de dato.

**3. Evaluación de completitud**  
Análisis de valores nulos por liga, cobertura de casas de apuestas y métricas globales de completitud.

**4. Validaciones de integridad**  
Comprobaciones de coherencia lógica y estructural (partidos por temporada, resultados, duplicados, valores negativos).

**5. Conclusiones del análisis exploratorio**  
Síntesis estructural y consideraciones para la fase de limpieza.

**6. Exportación del core multi-league**  
Guardado del core multi-league validado para las fases posteriores del pipeline.

## 0) Entorno y configuración

En esta sección se configuran las dependencias, librerías y parámetros globales necesarios para garantizar la reproducibilidad del análisis.

In [40]:
from pathlib import Path
import pandas as pd
import sys
import json
from IPython.display import display, Markdown

# Configuración de rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CORE_UNIFIED_PATH = PROCESSED_ROOT / "core_unified_raw.parquet"
METADATA_PATH = PROCESSED_ROOT / "core_unified_schema.json"

# Ligas a comparar y core dataset
LEAGUES = ["laliga", "premier", "bundesliga"]
PARQUET_NAME = "core_raw.parquet"  

# Importación de funciones propias
from src.analysis import group_columns

## 1) Lectura y organización de datos

Carga de los datasets core procesados de cada liga (formato Parquet).

In [41]:
dfs = {
    lg: pd.read_parquet(PROCESSED_ROOT / lg / PARQUET_NAME)
    for lg in LEAGUES
}

df_all = pd.concat(dfs.values(), ignore_index=True)

print(f"Datasets cargados: {len(dfs)}\n")
for lg in LEAGUES:
    rel_path = Path("data") / "processed" / lg / PARQUET_NAME
    print(f"  • {lg}: {rel_path}")

print(f"\nTotal de partidos: {sum(len(df) for df in dfs.values()):,}")

Datasets cargados: 3

  • laliga: data/processed/laliga/core_raw.parquet
  • premier: data/processed/premier/core_raw.parquet
  • bundesliga: data/processed/bundesliga/core_raw.parquet

Total de partidos: 10,661


## 2) Comparación de esquemas

Se compara la estructura de los datasets de las tres ligas para identificar qué columnas son comunes, cuáles son específicas de cada liga y si los tipos de datos se mantienen consistentes.

### 2.1 Dimensiones de los datasets

Comparación del número de partidos y columnas por liga.

In [42]:
dim_summary = pd.DataFrame([
    {
        "Liga": lg.capitalize(),
        "Partidos": len(df),
        "Columnas": len(df.columns)   
    }
    for lg, df in dfs.items()
])

display(dim_summary.style.hide(axis="index"))

Liga,Partidos,Columnas
Laliga,3800,43
Premier,3801,44
Bundesliga,3060,43


### 2.2 Identificación del core unificado

Variables comunes a las tres ligas que conforman el core mutli-league.

In [43]:
column_sets = {
    lg: set(df.columns)
    for lg, df in dfs.items()
}

core_unified = sorted(set.intersection(*column_sets.values()))
all_columns = sorted(set.union(*column_sets.values()))
league_specific = {
    lg: sorted(cols - set(core_unified))
    for lg, cols in column_sets.items()
}

print(f"Columnas totales (union de todas): {len(all_columns)}")
print(f"Columnas en core unificado: {len(core_unified)}\n")

has_specific = any(len(cols) > 0 for cols in league_specific.values())

if has_specific:
    print("Columnas específicas por liga:\n")
    for lg, specific in league_specific.items():
        if specific:
            print(f"  • {lg.capitalize()}: {', '.join(specific)}")
else:
    print("Todas las columnas son comunes a las 3 ligas")

Columnas totales (union de todas): 44
Columnas en core unificado: 43

Columnas específicas por liga:

  • Premier: Referee


### 2.3 Consistencia de tipos de datos

Verificación de que las columnas del core multi-league mantienen el mismo tipo en las tres ligas.

In [44]:
drift_cross_league = []

for col in core_unified:
    types_per_league = {lg: str(dfs[lg][col].dtype) for lg in LEAGUES}
    unique_types = set(types_per_league.values())
    
    if len(unique_types) > 1:
        drift_cross_league.append({
            "Column": col,
            **types_per_league
        })

drift_df = pd.DataFrame(drift_cross_league)

if len(drift_df) > 0:
    print(f"[!] Columnas del core con drift de tipos entre ligas: {len(drift_df)}\n")
    display(drift_df.set_index("Column"))
else:
    print("Tipos de datos consistentes en el core unificado")

[!] Columnas del core con drift de tipos entre ligas: 16



,laliga,premier,bundesliga
Column,,,
AC,int64,float64,int64
AF,int64,float64,int64
AR,int64,float64,int64
AS,int64,float64,int64
AST,int64,float64,int64
AY,int64,float64,int64
FTAG,int64,float64,int64
FTHG,int64,float64,int64
HC,int64,float64,int64


### 2.4 Resumen del core unificado

Clasificación y detalle de las variables comunes a las tres ligas.

#### Resumen global

In [45]:
core_unified_df = pd.DataFrame({"Variable": core_unified})

df_class = group_columns(core_unified_df["Variable"])

summary_groups = (
    df_class.groupby("Grupo", as_index=False)
    .agg(**{"Número de variables": ("Variable", "count")})
    .sort_values("Número de variables", ascending=False)
)

print(f"Core unificado: {len(core_unified)} variables comunes a las 3 ligas\n")
display(summary_groups.style.hide(axis="index"))

Core unificado: 43 variables comunes a las 3 ligas



Grupo,Número de variables
Cuotas de apuestas,21
Estadísticas del partido,12
Resultados y goles,6
Identificación del partido,4


#### Detalle de variables por grupo

In [46]:
detail_groups = (
    df_class.sort_values(["Grupo", "Variable"])
    .groupby("Grupo", as_index=False)
    .agg(Variables=("Variable", lambda x: "\n".join(x)))
)

display(
    detail_groups.style
    .set_properties(**{"white-space": "pre-wrap"})
    .hide(axis="index")
)

Grupo,Variables
Cuotas de apuestas,B365A B365D B365H BWA BWD BWH IWA IWD IWH PSA PSCA PSCD PSCH PSD PSH VCA VCD VCH WHA WHD WHH
Estadísticas del partido,AC AF AR AS AST AY HC HF HR HS HST HY
Identificación del partido,AwayTeam Date Div HomeTeam
Resultados y goles,FTAG FTHG FTR HTAG HTHG HTR


## 3) Evaluación de completitud

Evaluación de la calidad del core multi-league a través del análisis de valores nulos por liga, detección de casas de apuestas estables y síntesis de métricas globales de completitud.

### 3.1 Análisis de nulos por liga

Comparación del porcentaje de valores nulos en el core unificado entre las tres ligas.

In [47]:
null_summary = []

for lg, df in dfs.items():
    df_core = df[core_unified]
    total_cells = df_core.size
    total_nulls = df_core.isnull().sum().sum()
    null_pct = (total_nulls / total_cells) * 100
    
    null_summary.append({
        "Liga": lg.capitalize(),
        "Total nulos": total_nulls,
        "% nulos": round(null_pct, 2)
    })

null_df = pd.DataFrame(null_summary)
display(null_df.style.hide(axis="index"))

print(f"\nNulos totales en core unificado: {null_df['Total nulos'].sum():,}")

Liga,Total nulos,% nulos
Laliga,633,0.390000
Premier,601,0.370000
Bundesliga,534,0.410000



Nulos totales en core unificado: 1,768


### 3.2 Columnas con mayor proporción de nulos

Identificación de variables del core con completitud reducida por liga.

#### Construcción de la comparativa

In [48]:
null_data = {}

for lg, df in dfs.items():
    df_core = df[core_unified]
    col_nulls = df_core.isnull().sum()
    col_pct = (col_nulls / len(df_core)) * 100
    
    null_data[lg] = {
        col: col_pct[col]
        for col in core_unified
        if col_nulls[col] > 0
    }

all_null_cols = sorted(set().union(*[set(d.keys()) for d in null_data.values()]))

cols_in_all = [col for col in all_null_cols if all(col in null_data.get(lg, {}) for lg in LEAGUES)]
cols_partial = [col for col in all_null_cols if col not in cols_in_all]
ordered_cols = cols_in_all + cols_partial

print(f"Variables con nulos en al menos una liga: {len(all_null_cols)}")
print(f"  · Comunes a todas las ligas: {len(cols_in_all)}")
print(f"  · Específicas a alguna liga: {len(cols_partial)}")

Variables con nulos en al menos una liga: 43
  · Comunes a todas las ligas: 12
  · Específicas a alguna liga: 31


#### Visualización comparativa

In [49]:
rows = []
for col in ordered_cols:
    row = {}
    for lg in LEAGUES:
        lg_cap = lg.capitalize()
        if col in null_data.get(lg, {}):
            row[f"{lg_cap}_Variable"] = col
            row[f"{lg_cap}_%"] = f"{null_data[lg][col]:.2f}%"
        else:
            row[f"{lg_cap}_Variable"] = ""
            row[f"{lg_cap}_%"] = ""
    rows.append(row)

null_comparison_df = pd.DataFrame(rows)

null_comparison_df.columns = pd.MultiIndex.from_tuples(
    [(lg.capitalize(), "Variable") if "_Variable" in col else (lg.capitalize(), "% nulo") 
     for lg in LEAGUES for col in [f"{lg.capitalize()}_Variable", f"{lg.capitalize()}_%"]]
)

display(
    null_comparison_df
    .style
    .hide(axis="index")
    .set_table_attributes('style="max-height:400px; overflow-y:auto; display:block;"')
)

### 3.3 Casas de apuestas estables en el core multi-league

Filtrado de variables de casas de apuestas del core multi-league según umbral máximo de valores nulos.

In [50]:
# Prefijos de casas de apuestas según documentación oficial de football-data
bookmaker_prefixes = [
    "1XB", "B365", "BF", "BFD", "BMGM", "BV", "BS", "BW", 
    "CL", "GB", "IW", "LB", "PS", "SO", "SB", "SJ", 
    "SY", "VC", "WH"
]

odds_columns_core = [col for col in core_unified 
                     if any(col.startswith(book) for book in bookmaker_prefixes)]

THRESHOLD = 5.0
stable_bookmakers = []

for book in bookmaker_prefixes:
    book_cols = [c for c in odds_columns_core if c.startswith(book)]
    if not book_cols:
        continue
    
    max_null_pct = max(
        (dfs[lg][book_cols].isnull().sum().sum() / dfs[lg][book_cols].size) * 100
        for lg in LEAGUES
    )
    
    if max_null_pct <= THRESHOLD:
        stable_bookmakers.append({
            "Casa": book,
            "Columnas": len(book_cols),
            "Máx. % nulos": f"{max_null_pct:.2f}%",
            "Variables": ", ".join(sorted(book_cols))
        })

stable_df = pd.DataFrame(stable_bookmakers)

print(f"Umbral: ≤{THRESHOLD}% nulos | Casas de apuestas estables: {len(stable_df)}\n")
display(stable_df.style.hide(axis="index")) if len(stable_df) > 0 else print("Ninguna casa cumple el umbral")

Umbral: ≤5.0% nulos | Casas de apuestas estables: 5



Casa,Columnas,Máx. % nulos,Variables
B365,3,0.03%,"B365A, B365D, B365H"
BW,3,0.29%,"BWA, BWD, BWH"
PS,6,0.09%,"PSA, PSCA, PSCD, PSCH, PSD, PSH"
VC,3,0.03%,"VCA, VCD, VCH"
WH,3,0.03%,"WHA, WHD, WHH"


### 3.4 Resumen de completitud

Consolidación de hallazgos sobre la calidad de datos del core multi-league.

In [51]:
total_cols = len(core_unified)
cols_with_nulls = len(all_null_cols)
cols_clean = total_cols - cols_with_nulls

total_odds_cols = len(odds_columns_core)
stable_odds_cols = sum(stable_df["Columnas"])

print("RESUMEN DE COMPLETITUD DEL CORE UNIFICADO\n")
print(f"Variables comunes a las 3 ligas: {total_cols}")
print(f"  • Sin nulos: {cols_clean} ({(cols_clean/total_cols)*100:.1f}%)")
print(f"  • Con nulos: {cols_with_nulls} ({(cols_with_nulls/total_cols)*100:.1f}%)\n")

print(f"Columnas de cuotas: {total_odds_cols}")
print(f"  • Casas estables (<={THRESHOLD}% nulos): {len(stable_df)}")
print(f"  • Columnas utilizables: {stable_odds_cols}/{total_odds_cols} ({(stable_odds_cols/total_odds_cols)*100:.1f}%)\n")

RESUMEN DE COMPLETITUD DEL CORE UNIFICADO

Variables comunes a las 3 ligas: 43
  • Sin nulos: 0 (0.0%)
  • Con nulos: 43 (100.0%)

Columnas de cuotas: 21
  • Casas estables (<=5.0% nulos): 5
  • Columnas utilizables: 18/21 (85.7%)



## 4) Validaciones de integridad

Se realizan comprobaciones básicas de coherencia lógica y estructural sobre el core multi-league.


### 4.1 Consistencia estructural por temporada

Verificación del número esperado de partidos por temporada y del rango temporal cubierto en cada liga.

In [52]:
for lg, df in dfs.items():
    df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
    df["Season"] = df["Date"].dt.to_period("Y-JUL").astype(str)
    dfs[lg] = df
    
    min_date = df["Date"].min().strftime("%d/%m/%Y")
    max_date = df["Date"].max().strftime("%d/%m/%Y")
    total = len(df)
    
    season_counts = df.groupby("Season").size()
    expected = season_counts.mode()[0] 
    inconsistent = season_counts[season_counts != expected]
    
    if len(inconsistent) > 0:
        print(f"{lg.capitalize()}: {total:,} partidos | {min_date} → {max_date}")
        print(f"  [!] Inconsistencia: esperado {expected} partidos/temporada")
        for season, count in inconsistent.items():
            print(f"      {season}: {count} partidos")
    else:
        print(f"{lg.capitalize()}: {total:,} partidos ({expected}/temporada) | {min_date} → {max_date}")

Laliga: 3,800 partidos (380/temporada) | 23/08/2014 → 26/05/2024
Premier: 3,801 partidos (380/temporada) | 16/08/2014 → 19/05/2024
Bundesliga: 3,060 partidos (306/temporada) | 22/08/2014 → 18/05/2024


### 4.2 Validación de resultados (FTR/HTR)

Se valida la consistencia entre resultados declarados y marcadores en el core multi-league.

In [53]:
issues = sum(
    len(dfs[lg][
        ((dfs[lg]["FTR"] == "H") & (dfs[lg]["FTHG"] <= dfs[lg]["FTAG"])) |
        ((dfs[lg]["FTR"] == "A") & (dfs[lg]["FTAG"] <= dfs[lg]["FTHG"])) |
        ((dfs[lg]["FTR"] == "D") & (dfs[lg]["FTHG"] != dfs[lg]["FTAG"]))
    ])
    for lg in LEAGUES
)

print(f"Validación FTR/HTR: {issues} inconsistencias detectadas" if issues > 0 else "Validación FTR/HTR: todas las ligas correctas")

Validación FTR/HTR: todas las ligas correctas


### 4.3 Detección de duplicados

Identificación de partidos duplicados por liga, utilizando como clave compuesta por `Date`, `HomeTeam` y `AwayTeam`.

In [54]:
duplicates_found = False

for lg, df in dfs.items():
    dups = df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"], keep=False)
    
    if dups.sum() > 0:
        print(f"[!] {lg.capitalize()}: {dups.sum()} filas duplicadas")
        display(df[dups][["Date", "HomeTeam", "AwayTeam", "FTR"]].head())
        duplicates_found = True

if not duplicates_found:
    print("Sin duplicados detectados en ninguna liga")

Sin duplicados detectados en ninguna liga


### 4.4 Control de valores negativos

Control de calidad para columnas numéricas del core multi-league donde no se esperan valores negativos.

In [55]:
neg_found = False

for lg, df in dfs.items():
    numeric_cols = df[core_unified].select_dtypes(include="number").columns
    neg_counts = df[numeric_cols].lt(0).sum()
    neg_counts = neg_counts[neg_counts > 0].sort_values(ascending=False)
    
    if not neg_counts.empty:
        neg_found = True
        print(f"[!] {lg.capitalize()}: valores negativos detectados")
        display(
            neg_counts.rename("Valores negativos")
                      .reset_index()
                      .rename(columns={"index": "Columna"})
                      .style.hide(axis="index")
        )

if not neg_found:
    print("Ningún valor negativo detectado en columnas numéricas del core unificado")

Ningún valor negativo detectado en columnas numéricas del core unificado


### 4.5 Resumen de validaciones

Síntesis de los controles de integridad aplicados al core multi-league.

In [56]:
total_partidos = sum(len(df) for df in dfs.values())

print("RESUMEN DE VALIDACIONES DEL CORE MULTI-LEAGUE\n")
print(f"Partidos totales validados: {total_partidos:,}")
print(f"  • Inconsistencias FTR/HTR: {issues}")
print(f"  • Duplicados detectados: {sum(df.duplicated(subset=['Date','HomeTeam','AwayTeam'], keep=False).sum() for df in dfs.values())}")
print(f"  • Valores negativos: {'detectados' if neg_found else 'ninguno'}\n")

RESUMEN DE VALIDACIONES DEL CORE MULTI-LEAGUE

Partidos totales validados: 10,661
  • Inconsistencias FTR/HTR: 0
  • Duplicados detectados: 0
  • Valores negativos: ninguno



## 5) Conclusiones del análisis exploratorio

El análisis comparativo cubre **10 temporadas** (2014/15–2023/24) de tres competiciones europeas, con un total de **10.661 partidos**: 3.800 de La Liga, 3.801 de la Premier League (fila vacía identificado en la primera EDA) y 3.060 de la Bundesliga (18 equipos frente a 20 en las otras dos ligas).

Las tres competiciones presentan **rangos temporales alineados**, con temporadas que comienzan en agosto y finalizan entre mayo y junio del año siguiente.

### Core multi-league

Se identificó un **core dataset de 43 variables comunes** a las tres ligas, con tipos de datos consistentes (sin drift entre competiciones). El core se estructura en:

| Grupo | Variables |
|-------|-----------|
| Identificación (4) | `Date`, `Div`, `HomeTeam`, `AwayTeam` |
| Resultados (6) | `FTHG`, `FTAG`, `FTR`, `HTHG`, `HTAG`, `HTR` |
| Estadísticas (12) | `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR` |
| Cuotas (21) | `B365`, `BW`, `IW`, `PS`, `VC`, `WH` (H/D/A + variantes) |

### Completitud

- Las variables de **resultados y estadísticas** presentan **completitud total** en las tres ligas.
- Los valores nulos se concentran exclusivamente en **columnas de cuotas**, con un impacto global inferior al 1%.
- Las casas de apuestas con cobertura estable (≤5% nulos) en las tres ligas son las mismas identificadas en los EDA individuales: **B365, BW, PS, VC y WH**.

### Integridad

- **Sin inconsistencias** en la validación FTR/HTR en ninguna liga.
- **Sin duplicados** detectados por clave compuesta (`Date`, `HomeTeam`, `AwayTeam`).
- **Sin valores negativos** en columnas numéricas del core.

### Viabilidad del enfoque multi-liga para limpieza

El core unificado de **43 variables** mantiene tipos consistentes, alta completitud y coherencia lógica en las tres competiciones. Los patrones de calidad detectados son idénticos entre ligas: nulos concentrados en cuotas, mismas casas estables (B365, BW, PS, VC, WH), validaciones de integridad superadas.

Esto permite aplicar un **pipeline de limpieza unificado** sobre el dataset multi-league, sin lógica específica por liga. La única diferencia estructural (306 vs 380 partidos/temporada) afecta al volumen, no al esquema.

**Estrategia de modelado:** Se optará por un **enfoque conjunto** inicial, utilizando `Div` como feature categórica. Esto maximiza el volumen de datos (10,661 partidos) y permite al modelo capturar tanto patrones universales del fútbol como particularidades de cada liga. Si el rendimiento resultara inconsistente por competición, se evaluará la segmentación en modelos específicos.

## 6) Exportación del core multi-league

### 6.1 Montaje del core multi-liga 

Concatenación de las tres ligas utilizando exclusivamente las columnas del core unificado.

In [57]:
for lg in LEAGUES:
    missing = set(core_unified) - set(dfs[lg].columns)
    if missing:
        raise ValueError(f"[!] {lg}: columnas faltantes en core: {missing}")

df_core_unified = pd.concat(
    [dfs[lg][core_unified] for lg in LEAGUES],
    ignore_index=True
)

print(f"Core unificado construido:")
print(f"  Filas: {len(df_core_unified):,} | Columnas: {len(core_unified)}")
print(f"  Ligas: {', '.join([lg.capitalize() for lg in LEAGUES])}")

Core unificado construido:
  Filas: 10,661 | Columnas: 43
  Ligas: Laliga, Premier, Bundesliga


### 6.2 Exportación a Parquet y metadatos

Guardado del core multi-league en formato Parquet con metadatos del esquema para fases posteriores.

In [58]:
df_core_unified.to_parquet(CORE_UNIFIED_PATH, index=False)

metadata = {
    "num_columns": len(core_unified),
    "num_rows": len(df_core_unified),
    "num_leagues": len(LEAGUES),
    "leagues": LEAGUES,
    "columns": core_unified,
    "dtypes": {col: str(df_core_unified[col].dtype) for col in core_unified},
    "matches_per_league": {lg: len(dfs[lg]) for lg in LEAGUES}
}

with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

core_rel = CORE_UNIFIED_PATH.relative_to(PROJECT_ROOT)
metadata_rel = METADATA_PATH.relative_to(PROJECT_ROOT)

print(f"Archivos guardados:")
print(f"  · Dataset → {core_rel}")
print(f"  · Metadatos → {metadata_rel}")

Archivos guardados:
  · Dataset → data/processed/core_unified_raw.parquet
  · Metadatos → data/processed/core_unified_schema.json
